### SCD Type 1

In [0]:
%sql
create table datamodelling.default.scdtyp1_source
(
  prd_id int,
  prd_name string,
  prd_cat string,
  processDate Date
)

In [0]:
%sql
insert into datamodelling.default.scdtyp1_source
values
(1,'prd1','cat1',current_date()),
(2,'prd2','cat2',current_date()),
(3,'prd3','cat3',current_date())

In [0]:
%sql
create table datamodelling.gold.scdtyp1_table
(
  prd_id int,
  prd_name string,
  prd_cat string,
  processDate date
)

In [0]:
spark.sql('select * from datamodelling.default.scdtyp1_source').createOrReplaceTempView('src')

In [0]:
%sql
merge into datamodelling.gold.scdtyp1_table as trg
using src
on src.prd_id = trg.prd_id
when matched and src.processDate >= trg.processDate then update set *
when not matched then insert *

In [0]:
%sql
select * from datamodelling.gold.scdtyp1_table

In [0]:
%sql
update datamodelling.default.scdtyp1_source set prd_cat ='category'
where prd_id = 3

### SCD Type 2

In [0]:
%sql
create table datamodelling.default.scdtyp2_source
(
  prd_id int,
  prd_name string,
  prd_cat string,
  processDate Date
)

In [0]:
%sql
insert into datamodelling.default.scdtyp2_source
values
(1,'prd1','cat1',current_date()),
(2,'prd2','cat2',current_date()),
(3,'prd3','cat3',current_date())

In [0]:
%sql
create table datamodelling.gold.scdtyp2_table
(
  prd_id int,
  prd_name string,
  prd_cat string,
  processDate Date,
  start_date date,
  end_date date,
  in_use string
)

In [0]:
%sql
select *, current_date() as start_date, cast('3000-01-01' as date) as end_date,
'Y' as in_use 
from datamodelling.default.scdtyp2_source

In [0]:
spark.sql("""select *, current_date() as start_date, cast('3000-01-01' as date) as end_date,
'Y' as in_use 
from datamodelling.default.scdtyp2_source""").createOrReplaceTempView('scdtyp2_src')

In [0]:
%sql
merge into datamodelling.gold.scdtyp2_table as trg
using scdtyp2_src
on scdtyp2_src.prd_id = trg.prd_id
and trg.in_use ='Y' 
when matched and (
    scdtyp2_src.prd_name <> trg.prd_name or 
    scdtyp2_src.prd_cat <> trg.prd_cat or 
    scdtyp2_src.processDate <> trg.processDate
) then update set trg.end_date = current_date(),
  trg.in_use ='N'
when not matched then insert (
        prd_id,
        prd_name,
        prd_cat,
        processDate,
        start_date,
        end_date,
        in_use
)
values
(
        scdtyp2_src.prd_id,
        scdtyp2_src.prd_name,
        scdtyp2_src.prd_cat,
        scdtyp2_src.processDate,
        current_date(),
        null,
        'Y'
)

In [0]:
%sql
select * from datamodelling.gold.scdtyp2_table

In [0]:
%sql
update datamodelling.default.scdtyp2_source
set prd_cat = 'newCategory'
where prd_id = 3